# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Inspect methodology like an ML engineer; rewrite claims to the honest ladder.


## 1. Two paper findings + my methodology questions

Using FlyRank's public research framing (AI-driven SEO in numbers) as the external paper lens:

1. **Finding idea:** aggregate search metrics shift over time across large content corpora.
   **My question:** Was the unit page-day or page-snapshot? Did splits leak future windows?
2. **Finding idea:** AI referral / engagement shares are uneven and sparse.
   **My question:** Were zeros true zeros or unavailable GA4 fills? (`ga4_data_available` style flags).

My own work must answer the same questions for the refresh lane.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

results = json.loads((ROOT/'outputs'/'model_results.json').read_text())
print('My validation design:', results['split_strategy'])
print('Why: grouped by client_id — pages from the same client do not span train and test.')


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
My validation design: client_holdout
Why: grouped by client_id — pages from the same client do not span train and test.


## 2. My model under an honest split (before/after)

Client-holdout metrics (the honest split). Full-data top-K precision from the baseline notebook
is **not** the model selection number — it is only a descriptive queue read.


In [2]:
print('Honest (client-holdout) Precision@50:')
for k,v in results['models'].items():
    print(f"  {k}: {v['precision_at_50']:.3f} | ROC-AUC {v['roc_auc']:.3f}")
print(f"  baseline_rules: {results['baseline']['baseline_precision_at_50']:.3f}")
print(f"Base rate: {results['target_positive_rate']:.3f}")


Honest (client-holdout) Precision@50:
  decision_tree: 0.620 | ROC-AUC 0.742
  logistic_regression: 0.400 | ROC-AUC 0.700
  random_forest: 0.740 | ROC-AUC 0.750
  baseline_rules: 0.240
Base rate: 0.542


## 3. Leakage audit

Checklist executed against the feature lists in `scripts/ml_utils.py`.


In [3]:
sys.path.insert(0, str(ROOT/'scripts'))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
forbidden = {'trend_direction','trend_pct','is_declining_label','content_id','client_id'}
used = set(MODEL_NUMERIC_FEATURES)|set(MODEL_CATEGORICAL_FEATURES)
print('Leakage intersection with feature lists:', used & forbidden or 'NONE')
print('Label:', results['target'])
print('Audit: trend_* excluded; IDs excluded; client-holdout used.')
audit = {
    'forbidden_in_features': list(used & forbidden),
    'split': results['split_strategy'],
    'selection_metric': results['best_model']['selection_metric'],
}
(WORK_OUT/'w06_leakage_audit.json').write_text(json.dumps(audit, indent=2))


Leakage intersection with feature lists: NONE
Label: is_declining_label
Audit: trend_* excluded; IDs excluded; client-holdout used.


103

## 4. Claim rewrite

| Weak claim | Honest rewrite |
|---|---|
| "We predict Google rankings." | "We rank pages for refresh review using observed GSC/GA-style signals." |
| "RF is 74% accurate at finding declines." | "On a client-holdout slice of the starter set, RF Precision@50 was 0.74 vs baseline 0.24; base rate ≈0.54." |
| "Refreshing these pages will recover traffic." | "These pages are prioritized for human review; recovery is untested without an experiment." |


In [4]:
rewrites = [
    'Decision-support ranked queue — not auto-publishing.',
    'Metrics reported with base rate and baseline on the same split.',
    'No causal recovery claim without a designed experiment.',
]
print('\n'.join(f'- {r}' for r in rewrites))
(WORK_OUT/'w06_claim_rewrites.json').write_text(json.dumps(rewrites, indent=2))


- Decision-support ranked queue — not auto-publishing.
- Metrics reported with base rate and baseline on the same split.
- No causal recovery claim without a designed experiment.


195

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
